##### Applied image transformations check list:

- Center Crop -> **DONE**
- Rotate -> **DONE** (HOWEVER TRY TO MAKE IT SIMILAR TO MY VERSION)
- Translate/Shift -> **DONE**
- Scale -> **DONE**
- Shear -> **DONE**
- Gaussian Blur -> **DONE**
- Vertical Flip -> **DONE**
- Horizontal Flip -> **DONE**
- Gamma -> **DONE**
- Slice Replication 3D Filter -> **DONE**

# TEST SINGLE AUGMENTATIONS ON IMAGES

1 options for testing:

-> Individual testing: Load single image and apply transformations

In [ ]:
from math import ceil

import numpy as np
import albumentations as A
import matplotlib.pyplot as plt

from Scripts.NeuralNetworkSegmentation.transforms import *
from Scripts.NeuralNetworkSegmentation.data import read_input, read_mask

In [ ]:
IMAGE_PATH = "/home/Test_augmentations/B.e2024.21394_positive_0001_AA01_16_(814, 1070, 927, 1183).tif"
MASK_PATH = "/home/Test_augmentations/B.e2024.21394_positive_0001_AA01_16_(814, 1070, 927, 1183)_mask.tif"

CHANNEL = None # TODO: PERHAPS IMPLEMENT SELECTION OF CHANNEL SO ONLY ONE IS APPLIED or perhaps multiple

N_COLS = 4 # Number of columns that will be displaying images

PRINT_ORIGINALS = True # Print image and mask separately
PRINT_OVERLAP = True # Print mask overlayed on the image

In [ ]:
def print_info(image):
    print(f"""
    Shape: {image.shape}
    Type: {image.dtype}
    Values range: {np.unique(image)}
    """)

In [ ]:
def plot_each_depth_layer(image, mask = None): # TODO: Maybe instead of displaying across N_COLS use the number of channels?
    
    # transpose back from the albumentations required shape
    image = np.transpose(image, (0, 3, 1, 2))

    if mask is not None:
        mask = np.transpose(mask, (0, 3, 1, 2))

    n_depth_layers = image.shape[0]
    n_rows = ceil(n_depth_layers / N_COLS)

    fig, axes = plt.subplots(n_rows, N_COLS, figsize = (5 * N_COLS, 5 * n_rows), squeeze=False)
    fig.subplots_adjust(left=0.01, right=0.99, bottom=0.01, top=0.95, wspace=0.02, hspace=0.15)

    i = 0
    for row in axes:

        for col in row:
            
            if i < n_depth_layers:
                single_z = image[i][0] # Selects 1st channel (0-index) and slice i
                single_z = np.squeeze(single_z)

                col.imshow(single_z, cmap = "gray", aspect="equal")

                if mask is not None:
                    mask_single_z = mask[i][0] # Selects 1st channel (0-index) and slice i
                    col.imshow(mask_single_z, cmap="viridis", alpha=0.5)

                col.set_title(f"Slide {i+1}")
                col.axis("off")
            
            else:
                col.axis("off")

            i += 1
    
    # fig.tight_layout()
    fig.show()

In [ ]:
def show_transform_result(image_to_show, mask_to_show):
    if PRINT_ORIGINALS:
        print("IMAGE\n")
        print_info(image_to_show)
        plot_each_depth_layer(image_to_show)

        print("MASK\n")
        print_info(mask_to_show)
        plot_each_depth_layer(mask_to_show)

    if PRINT_OVERLAP:
        print("OVERLAP\n")
        plot_each_depth_layer(image_to_show, mask_to_show)

In [ ]:
image = read_input(IMAGE_PATH)
mask = read_mask(MASK_PATH)

# Convert to desired shape for albumentations: D,C,H,W -> D,H,W,C
image = np.transpose(image, (0, 2, 3, 1))
mask = np.transpose(mask, (0, 2, 3, 1))

show_transform_result(image, mask)

## ToFloat (Albumentations)

In [ ]:
tofloat = A.ToFloat(max_value=65535, p=1.0)
augmented = tofloat(volume=image)
image = augmented["volume"]

tofloat = A.ToFloat(max_value=255, p=1.0)
augmented = tofloat(mask3d=mask)
mask = augmented["mask3d"]

show_transform_result(image, mask)

## Center Crop (Albumentations)

In [ ]:
center_crop = A.CenterCrop(128, 128)

augmented = center_crop(volume=image, mask3d=mask)

aug_img = augmented["volume"]
aug_mask = augmented["mask3d"]

show_transform_result(aug_img, aug_mask)

## Rotate (Albumentations)

In [ ]:
rotate = A.Affine(
    rotate=(-180, 180),
    p=1.0,
)

augmented = rotate(volume=image, mask3d=mask)

aug_img = augmented["volume"]
aug_mask = augmented["mask3d"]

show_transform_result(aug_img, aug_mask)

## Translate/Shift (Albumentations)

In [ ]:
shift = A.Affine(
    translate_percent=(-0.2, 0.2),
    p=1.0,
)

augmented = shift(volume=image, mask3d=mask)

aug_img = augmented["volume"]
aug_mask = augmented["mask3d"]

show_transform_result(aug_img, aug_mask)

## Scale (Albumentations)

In [ ]:
scale = A.Affine(
    scale=(0.8, 1.2),
    keep_ratio=False,
    p=1.0,
)

augmented = scale(volume=image, mask3d=mask)

aug_img = augmented["volume"]
aug_mask = augmented["mask3d"]

show_transform_result(aug_img, aug_mask)

## Shear (Albumentations)

In [ ]:
shear = A.Affine(
    shear=(-3, 3),
    p=1.0,
)

augmented = shear(volume=image, mask3d=mask)

aug_img = augmented["volume"]
aug_mask = augmented["mask3d"]

show_transform_result(aug_img, aug_mask)

## Gaussian Blur (Albumentations)

In [ ]:
gaussian_blur = A.GaussianBlur(blur_range=[3, 5], sigma_range=[1.5, 3], p=1)

augmented = gaussian_blur(volume=image, mask3d=mask)

aug_img = augmented["volume"]
aug_mask = augmented["mask3d"]

show_transform_result(aug_img, aug_mask)

## Vertical Flip (Albumentations)

In [ ]:
vertical_flip = A.VerticalFlip(p=1)

augmented = vertical_flip(volume=image, mask3d=mask)

aug_img = augmented["volume"]
aug_mask = augmented["mask3d"]

show_transform_result(aug_img, aug_mask)

## Horizontal Flip (Albumentations)

In [ ]:
horizontal_flip = A.HorizontalFlip(p=1)

augmented = horizontal_flip(volume=image, mask3d=mask)

aug_img = augmented["volume"]
aug_mask = augmented["mask3d"]

show_transform_result(aug_img, aug_mask)

## Gamma (Albumentations)

In [ ]:
gamma = A.RandomGamma(gamma_range=[50, 100], p=1)

augmented = gamma(volume=image, mask3d=mask)

aug_img = augmented["volume"]
aug_mask = augmented["mask3d"]

show_transform_result(aug_img, aug_mask)

## Slice Replication then apply 3D filter

In [ ]:
increase_resolution = SliceReplication3DFilter(p=1)

augmented = increase_resolution(volume=image, mask3d=mask)

aug_img = augmented["volume"]
aug_mask = augmented["mask3d"]

show_transform_result(aug_img, aug_mask)

# Test Compose of Augmentations 

In [ ]:
transform_pipeline = A.Compose([
    
])